In [1]:
import numpy as np
import pandas as pd

import re

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity

/home/bittusah/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('archive/animes.csv')

In [3]:
df

,uid,title,synopsis,genre,aired,episodes,members,popularity,ranked,score,img_url,link
0,28891,Haikyuu!! Second Season,Following their participation at the Inter-Hig...,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun...","Oct 4, 2015 to Mar 27, 2016",25.0,489888,141,25.0,8.82,https://cdn.myanimelist.net/images/anime/9/766...,https://myanimelist.net/anime/28891/Haikyuu_Se...
1,23273,Shigatsu wa Kimi no Uso,Music accompanies the path of the human metron...,"['Drama', 'Music', 'Romance', 'School', 'Shoun...","Oct 10, 2014 to Mar 20, 2015",22.0,995473,28,24.0,8.83,https://cdn.myanimelist.net/images/anime/3/671...,https://myanimelist.net/anime/23273/Shigatsu_w...
2,34599,Made in Abyss,The Abyss—a gaping chasm stretching down into ...,"['Sci-Fi', 'Adventure', 'Mystery', 'Drama', 'F...","Jul 7, 2017 to Sep 29, 2017",13.0,581663,98,23.0,8.83,https://cdn.myanimelist.net/images/anime/6/867...,https://myanimelist.net/anime/34599/Made_in_Abyss
3,5114,Fullmetal Alchemist: Brotherhood,"""In order for something to be obtained, someth...","['Action', 'Military', 'Adventure', 'Comedy', ...","Apr 5, 2009 to Jul 4, 2010",64.0,1615084,4,1.0,9.23,https://cdn.myanimelist.net/images/anime/1223/...,https://myanimelist.net/anime/5114/Fullmetal_A...
4,31758,Kizumonogatari III: Reiketsu-hen,After helping revive the legendary vampire Kis...,"['Action', 'Mystery', 'Supernatural', 'Vampire']","Jan 6, 2017",1.0,214621,502,22.0,8.83,https://cdn.myanimelist.net/images/anime/3/815...,https://myanimelist.net/anime/31758/Kizumonoga...
...,...,...,...,...,...,...,...,...,...,...,...,...
19306,32979,Flip Flappers,Cocona is an average middle schooler living wi...,"['Sci-Fi', 'Adventure', 'Comedy', 'Magic']","Oct 6, 2016 to Dec 29, 2016",13.0,134252,843,1070.0,7.73,https://cdn.myanimelist.net/images/anime/4/822...,https://myanimelist.net/anime/32979/Flip_Flappers
19307,123,Fushigi Yuugi,"While visiting the National Library, junior-hi...","['Adventure', 'Fantasy', 'Magic', 'Martial Art...","Apr 6, 1995 to Mar 28, 1996",52.0,84407,1292,1071.0,7.73,https://cdn.myanimelist.net/images/anime/2/201...,https://myanimelist.net/anime/123/Fushigi_Yuugi
19308,1281,Gakkou no Kaidan,"Years ago, all of the ghosts in a haunted scho...","['Mystery', 'Horror', 'Supernatural']","Oct 22, 2000 to Mar 25, 2001",19.0,83093,1314,1073.0,7.73,https://cdn.myanimelist.net/images/anime/9/183...,https://myanimelist.net/anime/1281/Gakkou_no_K...
19309,450,InuYasha Movie 2: Kagami no Naka no Mugenjo,Inuyasha and company have finally destroyed Na...,"['Action', 'Adventure', 'Comedy', 'Historical'...","Dec 21, 2002",1.0,71989,1469,1077.0,7.73,https://cdn.myanimelist.net/images/anime/1162/...,https://myanimelist.net/anime/450/InuYasha_Mov...


In [4]:
df.shape

(19311, 12)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19311 entries, 0 to 19310
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   uid         19311 non-null  int64  
 1   title       19311 non-null  object 
 2   synopsis    18336 non-null  object 
 3   genre       19311 non-null  object 
 4   aired       19311 non-null  object 
 5   episodes    18605 non-null  float64
 6   members     19311 non-null  int64  
 7   popularity  19311 non-null  int64  
 8   ranked      16099 non-null  float64
 9   score       18732 non-null  float64
 10  img_url     19131 non-null  object 
 11  link        19311 non-null  object 
dtypes: float64(3), int64(3), object(6)
memory usage: 1.8+ MB


In [6]:
df['char_count'] = df['synopsis'].str.strip().str.len()
df['word_count'] = df['synopsis'].str.split().str.len()
df['sentence_count'] = df['synopsis'].str.count('[.!?]')

In [7]:
df[['char_count', 'word_count', 'sentence_count']].describe()

,char_count,word_count,sentence_count
count,18336.000000,18336.000000,18336.000000
mean,414.189954,69.583333,3.988165
std,351.572880,58.974403,3.406857
min,0.000000,0.000000,0.000000
25%,114.000000,19.000000,1.000000
50%,322.000000,54.000000,3.000000
75%,622.000000,105.000000,6.000000
max,2753.000000,466.000000,43.000000


In [8]:
df

,uid,title,synopsis,genre,aired,episodes,members,popularity,ranked,score,img_url,link,char_count,word_count,sentence_count
0,28891,Haikyuu!! Second Season,Following their participation at the Inter-Hig...,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun...","Oct 4, 2015 to Mar 27, 2016",25.0,489888,141,25.0,8.82,https://cdn.myanimelist.net/images/anime/9/766...,https://myanimelist.net/anime/28891/Haikyuu_Se...,1041.0,161.0,5.0
1,23273,Shigatsu wa Kimi no Uso,Music accompanies the path of the human metron...,"['Drama', 'Music', 'Romance', 'School', 'Shoun...","Oct 10, 2014 to Mar 20, 2015",22.0,995473,28,24.0,8.83,https://cdn.myanimelist.net/images/anime/3/671...,https://myanimelist.net/anime/23273/Shigatsu_w...,838.0,141.0,5.0
2,34599,Made in Abyss,The Abyss—a gaping chasm stretching down into ...,"['Sci-Fi', 'Adventure', 'Mystery', 'Drama', 'F...","Jul 7, 2017 to Sep 29, 2017",13.0,581663,98,23.0,8.83,https://cdn.myanimelist.net/images/anime/6/867...,https://myanimelist.net/anime/34599/Made_in_Abyss,1214.0,212.0,11.0
3,5114,Fullmetal Alchemist: Brotherhood,"""In order for something to be obtained, someth...","['Action', 'Military', 'Adventure', 'Comedy', ...","Apr 5, 2009 to Jul 4, 2010",64.0,1615084,4,1.0,9.23,https://cdn.myanimelist.net/images/anime/1223/...,https://myanimelist.net/anime/5114/Fullmetal_A...,1421.0,227.0,11.0
4,31758,Kizumonogatari III: Reiketsu-hen,After helping revive the legendary vampire Kis...,"['Action', 'Mystery', 'Supernatural', 'Vampire']","Jan 6, 2017",1.0,214621,502,22.0,8.83,https://cdn.myanimelist.net/images/anime/3/815...,https://myanimelist.net/anime/31758/Kizumonoga...,1093.0,186.0,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19306,32979,Flip Flappers,Cocona is an average middle schooler living wi...,"['Sci-Fi', 'Adventure', 'Comedy', 'Magic']","Oct 6, 2016 to Dec 29, 2016",13.0,134252,843,1070.0,7.73,https://cdn.myanimelist.net/images/anime/4/822...,https://myanimelist.net/anime/32979/Flip_Flappers,1063.0,175.0,7.0
19307,123,Fushigi Yuugi,"While visiting the National Library, junior-hi...","['Adventure', 'Fantasy', 'Magic', 'Martial Art...","Apr 6, 1995 to Mar 28, 1996",52.0,84407,1292,1071.0,7.73,https://cdn.myanimelist.net/images/anime/2/201...,https://myanimelist.net/anime/123/Fushigi_Yuugi,631.0,101.0,5.0
19308,1281,Gakkou no Kaidan,"Years ago, all of the ghosts in a haunted scho...","['Mystery', 'Horror', 'Supernatural']","Oct 22, 2000 to Mar 25, 2001",19.0,83093,1314,1073.0,7.73,https://cdn.myanimelist.net/images/anime/9/183...,https://myanimelist.net/anime/1281/Gakkou_no_K...,734.0,123.0,8.0
19309,450,InuYasha Movie 2: Kagami no Naka no Mugenjo,Inuyasha and company have finally destroyed Na...,"['Action', 'Adventure', 'Comedy', 'Historical'...","Dec 21, 2002",1.0,71989,1469,1077.0,7.73,https://cdn.myanimelist.net/images/anime/1162/...,https://myanimelist.net/anime/450/InuYasha_Mov...,721.0,129.0,9.0


In [9]:
df = df.drop(columns=['uid', 'aired', 'img_url', 'link'])

In [10]:
df

,title,synopsis,genre,episodes,members,popularity,ranked,score,char_count,word_count,sentence_count
0,Haikyuu!! Second Season,Following their participation at the Inter-Hig...,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun...",25.0,489888,141,25.0,8.82,1041.0,161.0,5.0
1,Shigatsu wa Kimi no Uso,Music accompanies the path of the human metron...,"['Drama', 'Music', 'Romance', 'School', 'Shoun...",22.0,995473,28,24.0,8.83,838.0,141.0,5.0
2,Made in Abyss,The Abyss—a gaping chasm stretching down into ...,"['Sci-Fi', 'Adventure', 'Mystery', 'Drama', 'F...",13.0,581663,98,23.0,8.83,1214.0,212.0,11.0
3,Fullmetal Alchemist: Brotherhood,"""In order for something to be obtained, someth...","['Action', 'Military', 'Adventure', 'Comedy', ...",64.0,1615084,4,1.0,9.23,1421.0,227.0,11.0
4,Kizumonogatari III: Reiketsu-hen,After helping revive the legendary vampire Kis...,"['Action', 'Mystery', 'Supernatural', 'Vampire']",1.0,214621,502,22.0,8.83,1093.0,186.0,9.0
...,...,...,...,...,...,...,...,...,...,...,...
19306,Flip Flappers,Cocona is an average middle schooler living wi...,"['Sci-Fi', 'Adventure', 'Comedy', 'Magic']",13.0,134252,843,1070.0,7.73,1063.0,175.0,7.0
19307,Fushigi Yuugi,"While visiting the National Library, junior-hi...","['Adventure', 'Fantasy', 'Magic', 'Martial Art...",52.0,84407,1292,1071.0,7.73,631.0,101.0,5.0
19308,Gakkou no Kaidan,"Years ago, all of the ghosts in a haunted scho...","['Mystery', 'Horror', 'Supernatural']",19.0,83093,1314,1073.0,7.73,734.0,123.0,8.0
19309,InuYasha Movie 2: Kagami no Naka no Mugenjo,Inuyasha and company have finally destroyed Na...,"['Action', 'Adventure', 'Comedy', 'Historical'...",1.0,71989,1469,1077.0,7.73,721.0,129.0,9.0


In [11]:
def basic_cleaning(text):
    if pd.isna(text):
        return ''

    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z0-9?!\s]', '', text)
    text = text.strip()
    return text

df['clean_synopsis'] = df['synopsis'].apply(basic_cleaning)

In [12]:
df[['synopsis', 'clean_synopsis']]

,synopsis,clean_synopsis
0,Following their participation at the Inter-Hig...,following their participation at the interhigh...
1,Music accompanies the path of the human metron...,music accompanies the path of the human metron...
2,The Abyss—a gaping chasm stretching down into ...,the abyssa gaping chasm stretching down into t...
3,"""In order for something to be obtained, someth...",in order for something to be obtained somethin...
4,After helping revive the legendary vampire Kis...,after helping revive the legendary vampire kis...
...,...,...
19306,Cocona is an average middle schooler living wi...,cocona is an average middle schooler living wi...
19307,"While visiting the National Library, junior-hi...",while visiting the national library juniorhigh...
19308,"Years ago, all of the ghosts in a haunted scho...",years ago all of the ghosts in a haunted schoo...
19309,Inuyasha and company have finally destroyed Na...,inuyasha and company have finally destroyed na...


In [13]:
df= df.dropna(subset=['clean_synopsis'])

In [14]:
df['clean_synopsis'].duplicated().sum()

np.int64(4172)

In [15]:
df.drop_duplicates(subset=['clean_synopsis'], inplace=True)

In [16]:
df

,title,synopsis,genre,episodes,members,popularity,ranked,score,char_count,word_count,sentence_count,clean_synopsis
0,Haikyuu!! Second Season,Following their participation at the Inter-Hig...,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun...",25.0,489888,141,25.0,8.82,1041.0,161.0,5.0,following their participation at the interhigh...
1,Shigatsu wa Kimi no Uso,Music accompanies the path of the human metron...,"['Drama', 'Music', 'Romance', 'School', 'Shoun...",22.0,995473,28,24.0,8.83,838.0,141.0,5.0,music accompanies the path of the human metron...
2,Made in Abyss,The Abyss—a gaping chasm stretching down into ...,"['Sci-Fi', 'Adventure', 'Mystery', 'Drama', 'F...",13.0,581663,98,23.0,8.83,1214.0,212.0,11.0,the abyssa gaping chasm stretching down into t...
3,Fullmetal Alchemist: Brotherhood,"""In order for something to be obtained, someth...","['Action', 'Military', 'Adventure', 'Comedy', ...",64.0,1615084,4,1.0,9.23,1421.0,227.0,11.0,in order for something to be obtained somethin...
4,Kizumonogatari III: Reiketsu-hen,After helping revive the legendary vampire Kis...,"['Action', 'Mystery', 'Supernatural', 'Vampire']",1.0,214621,502,22.0,8.83,1093.0,186.0,9.0,after helping revive the legendary vampire kis...
...,...,...,...,...,...,...,...,...,...,...,...,...
19002,Naruto x UT,All-new animation offered throughout UNIQLO cl...,"['Action', 'Comedy', 'Super Power', 'Martial A...",1.0,34155,2382,1728.0,7.50,348.0,55.0,3.0,allnew animation offered throughout uniqlo clo...
19003,Miira no Kaikata,High school student Sora Kashiwagi is accustom...,"['Slice of Life', 'Comedy', 'Supernatural']",12.0,61459,1648,1727.0,7.50,735.0,115.0,6.0,high school student sora kashiwagi is accustom...
19004,Shinryaku!? Ika Musume,"After regaining her squid-like abilities, Ika ...","['Slice of Life', 'Comedy', 'Shounen']",12.0,67422,1547,1548.0,7.56,614.0,102.0,6.0,after regaining her squidlike abilities ika mu...
19005,Kingsglaive: Final Fantasy XV,"For years, the Niflheim Empire and the kingdom...",['Action'],1.0,41077,2154,1544.0,7.56,1088.0,188.0,8.0,for years the niflheim empire and the kingdom ...


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15139 entries, 0 to 19006
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           15139 non-null  object 
 1   synopsis        15138 non-null  object 
 2   genre           15139 non-null  object 
 3   episodes        14756 non-null  float64
 4   members         15139 non-null  int64  
 5   popularity      15139 non-null  int64  
 6   ranked          13711 non-null  float64
 7   score           14884 non-null  float64
 8   char_count      15138 non-null  float64
 9   word_count      15138 non-null  float64
 10  sentence_count  15138 non-null  float64
 11  clean_synopsis  15139 non-null  object 
dtypes: float64(6), int64(2), object(4)
memory usage: 1.5+ MB


In [18]:
df['clean_word_count'] = df['clean_synopsis'].str.split().str.len()

In [19]:
df[['word_count', 'clean_word_count']].quantile([0.25, 0.75])

,word_count,clean_word_count
0.25,19.0,19.0
0.75,100.0,100.0


In [20]:
df.columns

Index(['title', 'synopsis', 'genre', 'episodes', 'members', 'popularity',
       'ranked', 'score', 'char_count', 'word_count', 'sentence_count',
       'clean_synopsis', 'clean_word_count'],
      dtype='object')

In [21]:
df = df.drop(columns=['episodes', 'members', 'popularity', 'ranked', 'score'])

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15139 entries, 0 to 19006
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             15139 non-null  object 
 1   synopsis          15138 non-null  object 
 2   genre             15139 non-null  object 
 3   char_count        15138 non-null  float64
 4   word_count        15138 non-null  float64
 5   sentence_count    15138 non-null  float64
 6   clean_synopsis    15139 non-null  object 
 7   clean_word_count  15139 non-null  int64  
dtypes: float64(3), int64(1), object(4)
memory usage: 1.0+ MB


In [23]:
df = df.dropna()

In [24]:
df.shape

(15138, 8)

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15138 entries, 0 to 19006
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             15138 non-null  object 
 1   synopsis          15138 non-null  object 
 2   genre             15138 non-null  object 
 3   char_count        15138 non-null  float64
 4   word_count        15138 non-null  float64
 5   sentence_count    15138 non-null  float64
 6   clean_synopsis    15138 non-null  object 
 7   clean_word_count  15138 non-null  int64  
dtypes: float64(3), int64(1), object(4)
memory usage: 1.0+ MB


In [26]:
df = df.drop(columns=['synopsis', 'word_count', 'char_count'])
df.columns = ['title', 'genre', 'sentence_count', 'synopsis', 'word_count']

In [27]:
df

,title,genre,sentence_count,synopsis,word_count
0,Haikyuu!! Second Season,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun...",5.0,following their participation at the interhigh...,161
1,Shigatsu wa Kimi no Uso,"['Drama', 'Music', 'Romance', 'School', 'Shoun...",5.0,music accompanies the path of the human metron...,141
2,Made in Abyss,"['Sci-Fi', 'Adventure', 'Mystery', 'Drama', 'F...",11.0,the abyssa gaping chasm stretching down into t...,212
3,Fullmetal Alchemist: Brotherhood,"['Action', 'Military', 'Adventure', 'Comedy', ...",11.0,in order for something to be obtained somethin...,227
4,Kizumonogatari III: Reiketsu-hen,"['Action', 'Mystery', 'Supernatural', 'Vampire']",9.0,after helping revive the legendary vampire kis...,186
...,...,...,...,...,...
19002,Naruto x UT,"['Action', 'Comedy', 'Super Power', 'Martial A...",3.0,allnew animation offered throughout uniqlo clo...,55
19003,Miira no Kaikata,"['Slice of Life', 'Comedy', 'Supernatural']",6.0,high school student sora kashiwagi is accustom...,115
19004,Shinryaku!? Ika Musume,"['Slice of Life', 'Comedy', 'Shounen']",6.0,after regaining her squidlike abilities ika mu...,102
19005,Kingsglaive: Final Fantasy XV,['Action'],8.0,for years the niflheim empire and the kingdom ...,188


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15138 entries, 0 to 19006
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           15138 non-null  object 
 1   genre           15138 non-null  object 
 2   sentence_count  15138 non-null  float64
 3   synopsis        15138 non-null  object 
 4   word_count      15138 non-null  int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 709.6+ KB


In [29]:
df = df.reset_index(drop=True)

In [30]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [31]:
synopsis_embeddings = model.encode(df['synopsis'].tolist(), show_progress_bar=True)

Batches: 100%|██████████| 474/474 [00:07<00:00, 66.43it/s] 


In [32]:
mlb = MultiLabelBinarizer()
genre_vectors = mlb.fit_transform(df['genre'])

In [33]:
genre_weight = 0.15

embeddings = np.hstack([synopsis_embeddings, genre_vectors * genre_weight])

In [34]:
similarity_matrix = cosine_similarity(embeddings)

In [35]:
def recommend(movie_title, top_k=5):
    idx = df[df['title'] == movie_title].index[0]
    sim_scores = similarity_matrix[idx]
    top_indices = sim_scores.argsort()[::-1][1: top_k +1]

    return df.iloc[top_indices][['title', 'genre']].assign(similarity_score=sim_scores[top_indices])

In [36]:
df['title'].sample(5)

13895    SAI: Part 1 / Revolving... to the Core
11200                   Shakugan no Shana Movie
8119                             Universal Mind
10520     Kiki to Lala no Papa to Mama ni Aitai
13493                   Saber Marionette J to X
Name: title, dtype: object

In [37]:
recommend('Kimetsu no Yaiba')

,title,genre,similarity_score
488,Dororo,"['Action', 'Adventure', 'Historical', 'Demons'...",0.721624
5460,Owari no Seraph,"['Action', 'Military', 'Supernatural', 'Drama'...",0.715583
9140,Honoo no Mirage,"['Action', 'Historical', 'Supernatural', 'Dram...",0.713791
7185,Hakkenden: Touhou Hakken Ibun,"['Action', 'Demons', 'Supernatural', 'Fantasy'...",0.705219
1164,Isuca,"['Action', 'Comedy', 'Ecchi', 'Romance', 'Scho...",0.701439
